In [1]:
# ============================================================
# FULL END-TO-END NOTEBOOK (TOP -> BOTTOM)
# Statcast 2021-2025 parquet -> PA-labeled pitch-state dataset
# -> Train CatBoost GPU models (PA outcome + ball type)
# -> Interactive menu w/ RBI odds (approx) + ball type output
# ============================================================

# ----------------------------
# 0) Imports
# ----------------------------
import re
import numpy as np
import pandas as pd

from pybaseball import playerid_lookup

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

from catboost import CatBoostClassifier


# ----------------------------
# 1) Load parquet
# ----------------------------
PARQUET_PATH = "statcast_2021_2025.parquet"
raw = pd.read_parquet(PARQUET_PATH)

print("loaded raw:", raw.shape)
print("num cols:", len(raw.columns))


# ----------------------------
# 2) Build PA-labeled pitch-state dataset
# Targets:
#   - pa_outcome: {out, strikeout, walk, single, double, triple, hr}
#   - ball_type: {ground_ball, fly_ball, line_drive, popup, not_in_play}
# ----------------------------

# find pitch order column
sort_col = "pitch_number" if "pitch_number" in raw.columns else ("pitch_num" if "pitch_num" in raw.columns else None)
if sort_col is None:
    raise ValueError("Need 'pitch_number' or 'pitch_num' to order pitches within PA.")

needed = [
    "game_pk","at_bat_number",sort_col,
    "batter","pitcher",
    "pitch_type","balls","strikes","outs_when_up","inning",
    "stand","p_throws",
    "on_1b","on_2b","on_3b",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "bat_score","fld_score",
    "events","des"
]

has_bb_type = "bb_type" in raw.columns
if has_bb_type:
    needed.append("bb_type")

missing = [c for c in needed if c not in raw.columns]
if missing:
    raise ValueError(f"Missing columns in parquet: {missing}")

tmp = raw[needed].copy()

# runners
tmp["runner_1b"] = tmp["on_1b"].notna().astype(np.int8)
tmp["runner_2b"] = tmp["on_2b"].notna().astype(np.int8)
tmp["runner_3b"] = tmp["on_3b"].notna().astype(np.int8)
tmp.drop(columns=["on_1b","on_2b","on_3b"], inplace=True)

# run differential
tmp["run_diff"] = (tmp["bat_score"].fillna(0) - tmp["fld_score"].fillna(0)).astype(np.int16)
tmp.drop(columns=["bat_score","fld_score"], inplace=True)

# sort + final pitch per PA
tmp = tmp.sort_values(["game_pk","at_bat_number",sort_col])
last_pitch = tmp.groupby(["game_pk","at_bat_number"], as_index=False).tail(1).copy()

def map_pa_outcome(events, des):
    ev = None if pd.isna(events) else str(events)
    d = "" if pd.isna(des) else str(des).lower()

    if ev == "single": return "single"
    if ev == "double": return "double"
    if ev == "triple": return "triple"
    if ev == "home_run": return "hr"

    if ev in ["walk","intent_walk","hit_by_pitch"]:
        return "walk"

    if "strikeout" in d or ev in ["strikeout","strikeout_double_play"]:
        return "strikeout"

    if ev is not None:
        return "out"

    return "none"

def map_ball_type(bb_type, pa_outcome):
    if pa_outcome in ["walk","strikeout"]:
        return "not_in_play"
    if bb_type is None or pd.isna(bb_type):
        return "not_in_play"
    bt = str(bb_type).lower().strip()
    if bt in ["ground_ball","fly_ball","line_drive","popup"]:
        return bt
    return "not_in_play"

last_pitch["pa_outcome"] = [map_pa_outcome(e, d) for e, d in zip(last_pitch["events"], last_pitch["des"])]
last_pitch = last_pitch[last_pitch["pa_outcome"] != "none"].copy()

if has_bb_type:
    last_pitch["ball_type"] = [map_ball_type(bt, oc) for bt, oc in zip(last_pitch["bb_type"], last_pitch["pa_outcome"])]
else:
    last_pitch["ball_type"] = "not_in_play"

last_pitch = last_pitch[["game_pk","at_bat_number","pa_outcome","ball_type"]].copy()

# merge targets onto each pitch-state row
tmp = tmp.merge(last_pitch, on=["game_pk","at_bat_number"], how="inner")

# drop terminal fields
drop_cols = ["events","des"]
if has_bb_type and "bb_type" in tmp.columns:
    drop_cols.append("bb_type")
tmp.drop(columns=drop_cols, inplace=True)

# modeling features
features = [
    "pitcher","batter",
    "pitch_type",
    "balls","strikes",
    "outs_when_up","inning",
    "stand","p_throws",
    "release_speed","release_spin_rate","pfx_x","pfx_z","zone",
    "runner_1b","runner_2b","runner_3b",
    "run_diff",
]

# numeric cleanup
for c in ["release_speed","release_spin_rate","pfx_x","pfx_z","zone"]:
    tmp[c] = pd.to_numeric(tmp[c], errors="coerce")

tmp = tmp.dropna(subset=["pitch_type","stand","p_throws","release_speed","pfx_x","pfx_z","zone"]).copy()

df = tmp[features + ["pa_outcome","ball_type"]].copy()

print("modeling df:", df.shape)
print("\npa_outcome dist (%):")
print(df["pa_outcome"].value_counts(normalize=True).mul(100).round(2))
print("\nball_type dist (%):")
print(df["ball_type"].value_counts(normalize=True).mul(100).round(2))


# ----------------------------
# 3) Train CatBoost model for PA outcome (GPU)
# ----------------------------
X = df[features].copy()
y = df["pa_outcome"].copy()

cat_cols = ["pitcher","batter","pitch_type","stand","p_throws","zone"]
for c in cat_cols:
    X[c] = X[c].astype("string").fillna("NA")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=1200,
    learning_rate=0.05,
    depth=8,
    eval_metric="MultiClass",
    verbose=100,
    task_type="GPU",
    devices="0"
)

model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_test, y_test),
    use_best_model=True
)

probs = model.predict_proba(X_test)
print("pa_outcome logloss:", log_loss(y_test, probs, labels=model.classes_))
print("pa_outcome classes:", list(model.classes_))


# ----------------------------
# 4) Train CatBoost model for Ball Type (GPU)
# ----------------------------
Xb = df[features].copy()
yb = df["ball_type"].copy()

for c in cat_cols:
    Xb[c] = Xb[c].astype("string").fillna("NA")

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=0.2, random_state=42, stratify=yb
)

model_balltype = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=800,
    learning_rate=0.06,
    depth=8,
    verbose=100,
    task_type="GPU",
    devices="0"
)

model_balltype.fit(
    Xb_train, yb_train,
    cat_features=cat_cols,
    eval_set=(Xb_test, yb_test),
    use_best_model=True
)

print("ball_type classes:", list(model_balltype.classes_))


# ----------------------------
# 5) Helpers for matchup prediction
# ----------------------------
_player_cache = {}

def mlbam_id_from_name(name):
    name = name.strip()
    if name in _player_cache:
        return _player_cache[name]
    parts = name.split()
    if len(parts) < 2:
        raise ValueError("Use 'First Last' for player name.")
    first, last = parts[0], " ".join(parts[1:])
    t = playerid_lookup(last, first)
    if t.empty:
        raise ValueError(f"Could not find player: {name}")
    mlbam = int(t.iloc[0]["key_mlbam"])
    _player_cache[name] = mlbam
    return mlbam

def parse_count(count_str):
    if count_str is None:
        return 0, 0
    s = str(count_str).strip()
    m = re.match(r"^\s*([0-3])\s*-\s*([0-2])\s*$", s)
    if not m:
        raise ValueError("count must look like '3-1'")
    return int(m.group(1)), int(m.group(2))

def parse_runners(runners_str):
    if runners_str is None:
        return 0, 0, 0
    r = str(runners_str).strip()
    if r == "" or r.lower() in ["empty","none","0"]:
        return 0, 0, 0
    bases = set([x.strip() for x in re.split(r"[;, ]+", r) if x.strip() != ""])
    return int("1" in bases), int("2" in bases), int("3" in bases)

def parse_score(score_str):
    if score_str is None:
        return 0
    s = str(score_str).strip()
    m = re.match(r"^\s*(-?\d+)\s*:\s*(-?\d+)\s*$", s)
    if not m:
        raise ValueError("score must look like '5:4'")
    return int(m.group(1)) - int(m.group(2))

def parse_pitcher_blob(blob):
    parts = [p.strip() for p in str(blob).split(",")]
    name = parts[0]
    count = parts[1] if len(parts) >= 2 and parts[1] else None
    runners = parts[2] if len(parts) >= 3 and parts[2] else None
    return name, count, runners

def _mode(series):
    return series.value_counts(dropna=True).index[0]

def approx_rbi_prob(outcome_probs_pct, r1, r2, r3, outs_when_up):
    runners = int(r1) + int(r2) + int(r3)
    p = {k: v / 100.0 for k, v in outcome_probs_pct.items()}

    if runners == 0:
        return round(float(outcome_probs_pct.get("hr", 0.0)), 2)

    pr_hr = p.get("hr", 0.0)
    pr_walk_rbi = p.get("walk", 0.0) * (1.0 if (r1 and r2 and r3) else 0.0)
    pr_triple_rbi = p.get("triple", 0.0) * 0.95

    if r2 or r3:
        dbl_rate = 0.85
    elif r1:
        dbl_rate = 0.45
    else:
        dbl_rate = 0.0
    pr_double_rbi = p.get("double", 0.0) * dbl_rate

    if r3:
        sng_rate = 0.80
    elif r2:
        sng_rate = 0.35
    elif r1:
        sng_rate = 0.05
    else:
        sng_rate = 0.0
    pr_single_rbi = p.get("single", 0.0) * sng_rate

    if outs_when_up >= 2:
        out_rate = 0.0
    else:
        if r3:
            out_rate = 0.07 if outs_when_up == 0 else 0.10
        else:
            out_rate = 0.01 if outs_when_up == 0 else 0.015
    pr_out_rbi = p.get("out", 0.0) * out_rate

    pr_rbi = pr_hr + pr_walk_rbi + pr_triple_rbi + pr_double_rbi + pr_single_rbi + pr_out_rbi
    pr_rbi = min(pr_rbi, 1.0)
    return round(pr_rbi * 100.0, 2)

def _build_situation_samples(
    batter_id,
    pitcher_id,
    balls,
    strikes,
    runner_1b,
    runner_2b,
    runner_3b,
    run_diff,
    n=1000
):
    pool = df[(df["pitcher"] == pitcher_id) & (df["balls"] == balls) & (df["strikes"] == strikes)]
    if len(pool) < 500:
        pool = df[df["pitcher"] == pitcher_id]
    if len(pool) < 500:
        raise ValueError("Not enough historical pitches for that pitcher in dataset")

    samp = pool.sample(n=min(n, len(pool)), replace=(len(pool) < n), random_state=42).copy()

    samp["batter"] = batter_id
    samp["pitcher"] = pitcher_id

    samp["balls"] = balls
    samp["strikes"] = strikes
    samp["runner_1b"] = runner_1b
    samp["runner_2b"] = runner_2b
    samp["runner_3b"] = runner_3b
    samp["run_diff"] = run_diff

    b_hist = df[df["batter"] == batter_id]
    p_hist = df[df["pitcher"] == pitcher_id]
    if len(b_hist) > 0:
        samp["stand"] = _mode(b_hist["stand"])
    if len(p_hist) > 0:
        samp["p_throws"] = _mode(p_hist["p_throws"])

    samp = samp.fillna(0)

    # cat cols as strings for GPU predict
    for c in cat_cols:
        if c in samp.columns:
            samp[c] = samp[c].astype("string").fillna("NA")

    return samp[features]

def predict_matchup(
    batter_name,
    pitcher_blob,
    score=None,
    count=None,
    runners=None,
    outs=0,
    n_sims=1000
):
    p_name, p_count, p_runners = parse_pitcher_blob(pitcher_blob)
    if p_count is not None:
        count = p_count
    if p_runners is not None:
        runners = p_runners
    if count is None:
        count = "0-0"

    balls, strikes = parse_count(count)
    r1, r2, r3 = parse_runners(runners)
    rd = parse_score(score)

    batter_id = mlbam_id_from_name(batter_name)
    pitcher_id = mlbam_id_from_name(p_name)

    Xsim = _build_situation_samples(
        batter_id=batter_id,
        pitcher_id=pitcher_id,
        balls=balls,
        strikes=strikes,
        runner_1b=r1,
        runner_2b=r2,
        runner_3b=r3,
        run_diff=rd,
        n=n_sims
    ).copy()

    # force outs state
    Xsim["outs_when_up"] = outs

    # PA outcome
    prob_mat = model.predict_proba(Xsim)
    avg_probs = prob_mat.mean(axis=0)
    out_probs = pd.Series(avg_probs, index=model.classes_).sort_values(ascending=False)
    out_probs_pct = (out_probs * 100).round(2).to_dict()

    # Ball type
    bt_mat = model_balltype.predict_proba(Xsim)
    bt_avg = bt_mat.mean(axis=0)
    bt_probs = pd.Series(bt_avg, index=model_balltype.classes_).sort_values(ascending=False)
    bt_probs_pct = (bt_probs * 100).round(2).to_dict()

    # RBI odds (approx)
    rbi_pct = approx_rbi_prob(out_probs_pct, r1, r2, r3, outs)

    return {
        "batter": batter_name,
        "pitcher": p_name,
        "count": f"{balls}-{strikes}",
        "outs": outs,
        "runners": {"1b": r1, "2b": r2, "3b": r3},
        "run_diff": rd,
        "n_sims": n_sims,
        "probs_percent": out_probs_pct,
        "ball_type_percent": bt_probs_pct,
        "rbi_percent": rbi_pct
    }


# ----------------------------
# 6) Interactive Menu (ipywidgets)
# ----------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output

batter_in = widgets.Text(value="Aaron Judge", description="Batter:", layout=widgets.Layout(width="380px"))
pitcher_in = widgets.Text(value="Tarik Skubal", description="Pitcher:", layout=widgets.Layout(width="380px"))

balls_dd = widgets.Dropdown(options=[0,1,2,3], value=0, description="Balls:")
strikes_dd = widgets.Dropdown(options=[0,1,2], value=0, description="Strikes:")
outs_dd = widgets.Dropdown(options=[0,1,2], value=0, description="Outs:")

r1_cb = widgets.Checkbox(value=False, description="1B")
r2_cb = widgets.Checkbox(value=False, description="2B")
r3_cb = widgets.Checkbox(value=False, description="3B")

bat_score_in = widgets.IntText(value=0, description="Bat score:")
fld_score_in = widgets.IntText(value=0, description="Fld score:")

sims_sl = widgets.IntSlider(value=1000, min=200, max=5000, step=200, description="Sims:", continuous_update=False)

submit_btn = widgets.Button(description="Submit", button_style="success", icon="check")
out_box = widgets.Output()

def _runners_str():
    bases = []
    if r1_cb.value: bases.append("1")
    if r2_cb.value: bases.append("2")
    if r3_cb.value: bases.append("3")
    return ";".join(bases) if bases else None

def _score_str():
    return f"{bat_score_in.value}:{fld_score_in.value}"

def on_submit(_):
    with out_box:
        clear_output()

        batter = batter_in.value.strip()
        pitcher = pitcher_in.value.strip()

        count = f"{balls_dd.value}-{strikes_dd.value}"
        outs = int(outs_dd.value)

        runners = _runners_str()
        score = _score_str()
        n_sims = int(sims_sl.value)

        try:
            res = predict_matchup(
                batter,
                f"{pitcher}, {count}" + (f", {runners}" if runners else ""),
                score,
                outs=outs,
                n_sims=n_sims
            )

            # outcome table
            probs = res["probs_percent"]
            df_out = (
                pd.DataFrame({"Outcome": list(probs.keys()), "Percent": list(probs.values())})
                .sort_values("Percent", ascending=False)
                .reset_index(drop=True)
            )

            # ball type table
            bt = res["ball_type_percent"]
            df_bt = (
                pd.DataFrame({"BallType": list(bt.keys()), "Percent": list(bt.values())})
                .sort_values("Percent", ascending=False)
                .reset_index(drop=True)
            )

            print(f"{res['batter']} vs {res['pitcher']}")
            print(f"Count: {res['count']} | Outs: {res['outs']} | Runners: {res['runners']} | Run diff: {res['run_diff']} | Sims: {res['n_sims']}")
            print(f"Odds of ≥1 RBI (approx): {res['rbi_percent']}%")
            display(df_out)
            display(df_bt)

        except Exception as e:
            print("Error:", str(e))

submit_btn.on_click(on_submit)

row1 = widgets.HBox([batter_in, pitcher_in])
row2 = widgets.HBox([balls_dd, strikes_dd, outs_dd, widgets.Label("Runners:"), r1_cb, r2_cb, r3_cb])
row3 = widgets.HBox([bat_score_in, fld_score_in, sims_sl, submit_btn])

ui = widgets.VBox([row1, row2, row3, out_box])
display(ui)

# ----------------------------
# Optional quick manual test (uncomment)
# ----------------------------
# print(predict_matchup("Aaron Judge","Tarik Skubal, 3-1, 1;2","5:4",outs=1,n_sims=2000))

loaded raw: (3846144, 119)
num cols: 119
modeling df: (3741396, 20)

pa_outcome dist (%):
pa_outcome
out          39.80
strikeout    28.31
walk         13.03
single       12.16
double        3.72
hr            2.65
triple        0.33
Name: proportion, dtype: float64

ball_type dist (%):
ball_type
not_in_play    41.49
ground_ball    25.18
fly_ball       15.25
line_drive     13.94
popup           4.13
Name: proportion, dtype: float64
0:	learn: 1.8900844	test: 1.8901202	best: 1.8901202 (0)	total: 212ms	remaining: 4m 13s
100:	learn: 1.3859076	test: 1.3855145	best: 1.3855145 (100)	total: 9.37s	remaining: 1m 41s
200:	learn: 1.3791024	test: 1.3793661	best: 1.3793661 (200)	total: 18.5s	remaining: 1m 32s
300:	learn: 1.3753546	test: 1.3765873	best: 1.3765873 (300)	total: 27.6s	remaining: 1m 22s
400:	learn: 1.3726571	test: 1.3748487	best: 1.3748487 (400)	total: 36.5s	remaining: 1m 12s
500:	learn: 1.3700118	test: 1.3732401	best: 1.3732401 (500)	total: 45.7s	remaining: 1m 3s
600:	learn: 1.3678502	t

In [3]:
import json

# 1) Save models
model.save_model("model_pa.cbm")
model_balltype.save_model("model_balltype.cbm")

# 2) Save the slim df that _build_situation_samples() uses
# (You already have df = tmp[features + targets] so this is small enough vs raw Statcast)
df.to_parquet("model_df.parquet", index=False)

# 3) Save meta (features + categorical columns)
meta = {
    "features": features,
    "cat_cols": cat_cols
}
with open("model_meta.json", "w") as f:
    json.dump(meta, f)

print("Saved:")
print(" - model_pa.cbm")
print(" - model_balltype.cbm")
print(" - model_df.parquet")
print(" - model_meta.json")

Saved:
 - model_pa.cbm
 - model_balltype.cbm
 - model_df.parquet
 - model_meta.json
